# 013-refinement-cnn

# Imports

In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.eval import (HOLDOUT_LOCATION, MEAN_KEY, STRIP_LOCATION, STRIP_ROWS, VAL_SETS,
                                      bilinear_argmax, evaluate, load_baseline_model, load_val_sets,
                                      lraspp_outputs, results_table, run_model)
from uavloc.segmentation.latency import model_latency, refiner_latency, standard_tile
from uavloc.segmentation.refine.refinement_cnn import (RefinementCNN, RefinerConfig, TrainConfig, build_crop_bank,
                                                       is_finite, n_parameters, refine_fn, train_refiner)
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "013-refinement-cnn"
NOTE = ""
LOG_RUN = True
MODE = "refiner"                  # "refiner" | "baseline"
USE_EDGES = False
SEED = 42
WIDTH = 32
DILATIONS = [1, 2, 4]
UNSHUFFLE = 4
N_TRAIN_CROPS = 512
N_HOLDOUT_CROPS = 64
HOLDOUT_SEED = 123
BANK_CROP_PX = 512
BANK_BATCH = 8
RARE_CROP_P = 0.3
GAP_PX = 64
STEPS = 3000
BATCH = 16
CROP_PX = 256
LR = 3e-3
WEIGHT_DECAY = 1e-4
WARMUP_FRAC = 0.05
LOG_EVERY = 250
TRAIN_DEVICE = "mps"
EVAL_DEVICE = "cpu"
PROBE_RESULTS = "pocs/mobilenet-lraspp-speed/results-lraspp.json"
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_WARMUP = 5
LATENCY_RUNS = 30
LATENCY_THREADS = 4
LATENCY_BUDGET_RATIO = 0.3
ZOOM_PX = 256
ZOOMS = [["birky-strip-google", 0, 64, 256, "buildings"], ["birky-strip-google", 0, 192, 2304, "track"],
         ["bushcha", 0, 1664, 1472, "buildings"], ["bushcha", 0, 1728, 768, "track"]]

# Data

In [ ]:
set_seed(SEED)
train_device = torch.device(TRAIN_DEVICE if TRAIN_DEVICE != "mps" or torch.backends.mps.is_available() else "cpu")
refiner_config = RefinerConfig(width=WIDTH, dilations=tuple(DILATIONS), unshuffle=UNSHUFFLE, use_edges=USE_EDGES)
train_config = TrainConfig(steps=STEPS, batch=BATCH, crop_px=CROP_PX, lr=LR, weight_decay=WEIGHT_DECAY,
                           warmup_frac=WARMUP_FRAC, log_every=LOG_EVERY)
train_device

In [ ]:
val_sets = load_val_sets()
{name: len(images) for name, images in val_sets.items()}

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / "ua-village-annotated", DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sat_train = [s for s in sat if not s.banned.all()]
probe_splits = json.loads((REPO_ROOT / PROBE_RESULTS).read_text())["splits"]["sat_train"]
assert [(s.loc, s.date) for s in sat_train] == [tuple(p) for p in probe_splits]
[(s.loc.split("-")[0], s.date) for s in sat_train]

# Model

In [ ]:
segmenter = load_baseline_model(EVAL_DEVICE)

In [ ]:
cases = run_model(segmenter, val_sets, EVAL_DEVICE)

In [ ]:
refiner = RefinementCNN(refiner_config)
n_parameters(refiner)

# Run

In [ ]:
baseline = evaluate(bilinear_argmax, cases)
results_table(baseline).round(4)

In [ ]:
def train_banks() -> tuple:
    """Training and hold-out crops drawn by LR-ASPP's own sampler, with the frozen model's stride-8 logits."""
    sampler = vs.SatelliteSampler(sat_train, BANK_CROP_PX, RARE_CROP_P)
    bank_model = load_baseline_model(train_device)
    train = build_crop_bank(bank_model, sampler, N_TRAIN_CROPS, BANK_BATCH, SEED, train_device)
    holdout = build_crop_bank(bank_model, sampler, N_HOLDOUT_CROPS, BANK_BATCH, HOLDOUT_SEED, train_device)
    return train, holdout

In [ ]:
history, train_minutes = [], 0.0
if MODE == "refiner":
    start = time.time()
    train_bank, holdout_bank = train_banks()
    bank_minutes = (time.time() - start) / 60
    history = train_refiner(refiner, train_bank, holdout_bank, train_config, SEED, train_device)
    train_minutes = (time.time() - start) / 60 - bank_minutes
    refiner = refiner.cpu().eval()
pd.DataFrame(history).round(4)

In [ ]:
predict = refine_fn(refiner, EVAL_DEVICE) if MODE == "refiner" else bilinear_argmax
results = evaluate(predict, cases)
results_table(results).round(4)

In [ ]:
per_class = pd.concat({"baseline": pd.DataFrame({n: baseline[n]["biou_per_class"] for n in VAL_SETS}).T,
                       "refined": pd.DataFrame({n: results[n]["biou_per_class"] for n in VAL_SETS}).T}, axis=1)
per_class.round(3)

In [ ]:
latency_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(segmenter, latency_tile, EVAL_DEVICE)
timing = dict(warmup=LATENCY_WARMUP, runs=LATENCY_RUNS, threads=LATENCY_THREADS)
model_before = model_latency(segmenter, latency_tile, **timing)
refiner_time = refiner_latency(predict, latency_output, latency_tile, **timing)
model_after = model_latency(segmenter, latency_tile, **timing)
model_ms = (model_before.median_ms + model_after.median_ms) / 2
latency_ratio = refiner_time.median_ms / model_ms
{"model_ms": [model_before.median_ms, model_after.median_ms], "refiner_ms": refiner_time.median_ms,
 "ratio": latency_ratio, "within_budget": latency_ratio <= LATENCY_BUDGET_RATIO}

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
fig_curves, ax = plt.subplots(figsize=(6, 3.5))
if history:
    frame = pd.DataFrame(history)
    ax.plot(frame["step"], frame["train_ce"], "-o", label="train CE (256 px sub-crops)")
    ax.plot(frame["step"], frame["holdout_ce"], "-s", label="hold-out CE, refined")
    ax.plot(frame["step"], frame["holdout_ce_identity"], "--", label="hold-out CE, LR-ASPP alone")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax.set_title(f"edges={USE_EDGES} seed={SEED}")
ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_curves.tight_layout()

In [ ]:
def zoom_row(axes: np.ndarray, zoom: list) -> None:
    """RGB | label | baseline | refined for one `ZOOM_PX` window of a val image."""
    name, index, top, left, what = zoom
    case, win = cases[name][index], np.s_[top:top + ZOOM_PX, left:left + ZOOM_PX]
    panels = [(case.image.rgb[win], f"{name} {case.image.date} ({what})"), (colour(case.image.label[win]), "label"),
              (colour(bilinear_argmax(case.output, case.image.rgb)[win]), "LR-ASPP bilinear"),
              (colour(predict(case.output, case.image.rgb)[win]), f"{MODE}, edges={USE_EDGES}")]
    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")

In [ ]:
fig_zoom, axes = plt.subplots(len(ZOOMS), 4, figsize=(13, 3.3 * len(ZOOMS)))
for row, zoom in zip(axes, ZOOMS):
    zoom_row(row, zoom)
fig_zoom.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
                loc="lower center", ncol=len(CLASSES))
fig_zoom.tight_layout(rect=(0, .03, 1, 1))

# Log

In [ ]:
def run_metrics() -> dict:
    """Primary metric first (report.py takes the first scalar), then per set, deltas, per class, latency."""
    metrics = {"boundary_iou_mean": results[MEAN_KEY]["biou"], "miou_mean": results[MEAN_KEY]["miou"],
               "bf1_1px_mean": results[MEAN_KEY]["bf1_1px"], "bf1_2px_mean": results[MEAN_KEY]["bf1_2px"]}
    metrics.update({f"{k}_{n}": v for n in VAL_SETS for k, v in results[n].items() if not isinstance(v, dict)})
    metrics.update({f"delta_{k}_{n}": results[n][k] - baseline[n][k] for n in [*VAL_SETS, MEAN_KEY]
                    for k in ("miou", "biou", "bf1_1px", "bf1_2px")})
    metrics.update({f"biou_{c}_{n}": v for n in VAL_SETS for c, v in results[n]["biou_per_class"].items()})
    metrics.update({"refiner_ms_median": refiner_time.median_ms, "refiner_ms_p10": refiner_time.p10_ms,
                    "refiner_ms_p90": refiner_time.p90_ms, "model_ms_median": model_ms,
                    "refiner_to_model_ratio": latency_ratio, "n_parameters": n_parameters(refiner),
                    "train_minutes": train_minutes})
    if history:
        metrics.update({"holdout_ce": history[-1]["holdout_ce"], "holdout_ce_identity": history[-1]["holdout_ce_identity"],
                        "history": history})
    return metrics

In [ ]:
metrics = run_metrics()
RUN_CONFIG = {"mode": MODE, "refiner": asdict(refiner_config), "train": asdict(train_config),
              "n_train_crops": N_TRAIN_CROPS, "n_holdout_crops": N_HOLDOUT_CROPS, "holdout_seed": HOLDOUT_SEED,
              "bank_crop_px": BANK_CROP_PX, "rare_crop_p": RARE_CROP_P, "gap_px": GAP_PX,
              "segmenter": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt", "eval_device": EVAL_DEVICE,
              "latency_threads": LATENCY_THREADS, "latency_budget_ratio": LATENCY_BUDGET_RATIO}
{k: round(v, 4) for k, v in metrics.items() if isinstance(v, float) and ("mean" in k or "ms" in k or "ratio" in k)}

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
checkpoint = artifacts / f"refiner-edges{int(USE_EDGES)}-seed{SEED}.pt"
if MODE == "refiner":
    torch.save(refiner.state_dict(), checkpoint)
status = "ok" if is_finite(history) else "failed"
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_curves, fig_zoom], status=status,
              seed=SEED, checkpoint=str(checkpoint.relative_to(REPO_ROOT)) if MODE == "refiner" else None,
              device=str(train_device) if MODE == "refiner" else EVAL_DEVICE) if LOG_RUN else {"run": "dry"}
run["run"]